<a href="https://colab.research.google.com/github/acapodanno/openai-agent-sdk/blob/main/nick_distructor_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install pydantic

In [12]:
from pydantic import BaseModel,Field
class Article(BaseModel):
  title: str = Field(description="Title of the article")
  content: str  = Field(description="Content of the article. It should contain html tags")
  seo_title: str = Field(description="SEO title for Yoast (max 60 characters)")
  meta_description: str = Field(description="Meta description for Yoast (150–160 characters)")
  focus_keyphrase: str = Field(description="Primary focus keyphrase for Yoast SEO")

In [13]:
!pip install openai-agents

In [14]:
from agents import Agent, Runner,set_default_openai_key

#### Define Agent
async def run_agent(project_input):
  agent = Agent(
      name="Web3 Article Generator",
      instructions="""
You are a senior Web3 content expert and SEO writer.

Generate a factual, authoritative, SEO-optimized HTML guide about the given Web3 project.
Target audience: crypto-native users.
Tone: professional, neutral, informative.
No hype. No marketing language. No false promises.

The article length must be between 900 and 1,200 words.

Return ONLY a JSON object matching the Article schema with these fields:
- title
- content (HTML)
- seo_title
- meta_description
- focus_keyphrase

================================
INPUT INTERPRETATION RULES
================================

The input is free text and may include:
- A project name OR a project URL
- The project type or category
- A referral link (optional)

Interpretation rules:
- If a project URL is provided, infer the project name from the domain.
- If both a project URL and a referral link are provided, use the referral link.
- If only one link is provided, treat it as the referral link.
- Do not invent referral links.

================================
ABSOLUTE SEO CONSTRAINTS
================================

These rules are mandatory and must never be violated.

1. Forbidden characters:
- Hyphen (-)
- En dash (–)
- Em dash (—)

These characters are FORBIDDEN in:
- title
- seo_title
- meta_description
- focus_keyphrase
- article body text

If any forbidden character appears, regenerate the text until none remain.

2. Focus keyphrase:
- Plain text only
- No punctuation
- No special characters
- Maximum 4 or 5 words
- Must clearly describe the project

3. SEO title:
- Maximum 60 characters
- Colon (:) is allowed
- Must include the focus keyphrase naturally
- Clear and descriptive

4. Meta description:
- Maximum 155 characters
- Natural language sentence
- Include the focus keyphrase exactly once
- No emojis

================================
HTML RULES
================================

Use ONLY the following HTML tags:
- h1
- h2
- p
- ul
- li
- strong
- a

No markdown.
No div.
No inline styles.

================================
ARTICLE STRUCTURE
================================

- H1 title
- Introduction with a clear value proposition and soft CTA
- What the project is
- How it works step by step
- Technology overview
- Key features and advantages
- Use cases
- Why early users benefit
- Risks and limitations
- Final CTA

================================
LINKING RULES
================================

- If a referral link is not provided, do not insert any referral links.
- If a referral link is provided, insert it exactly 3 times.
- Do not place referral links inside headings
- Integrate links naturally into paragraphs

================================
FINAL VALIDATION
================================

Before returning the final JSON:
- Scan all fields for forbidden characters
- Ensure all constraints are respected
- Fix any violation automatically
""",
      model="gpt-5-mini",
      output_type=Article,
  )

  runner = Runner()
  return await runner.run(
      agent,
      f"""
Write a detailed, well structured article about the Web3 project described below.
Write in English.

PROJECT INPUT:
{project_input}

Follow EXACTLY the structure and rules defined in the instructions.
Return valid HTML content suitable for WordPress Elementor.
"""
  )


In [15]:
from google.colab import userdata
set_default_openai_key(userdata.get('OPENAI_API_KEY'))



In [16]:
from requests import post
baseUrl = userdata.get('WP_BASE_URL')
wp_username = userdata.get('WP_USERNAME')
wp_password = userdata.get('WP_APP_PASSWORD')
"zNcp MHGn CYCH RspY Ievf nldd"

def public_draft_wp(titolo: str, contenuto_html: str, seo_title: str, meta_description: str, focus_keyphrase: str) -> dict:
    endpoint = f"{baseUrl.rstrip('/')}/wp-json/wp/v2/posts"
    auth = (wp_username, wp_password)

    payload = {
        "title": titolo,
        "content": contenuto_html,
        "status": "draft",
        "meta": {
            "_yoast_wpseo_title": seo_title,
            "_yoast_wpseo_metadesc": meta_description,
            "_yoast_wpseo_focuskw": focus_keyphrase
        }
    }

    headers = {
        "Content-Type": "application/json"
    }

    response = post(endpoint, json=payload, auth=auth, headers=headers)

    if not response.ok:
        print("Errore WordPress:", response.status_code, response.text)
        response.raise_for_status()

    return response.json()


In [17]:
async def main():
  project_input = input( "Enter project (name or link), project type, and referral link if any:\n")
  result = await run_agent(project_input)
  article = result.final_output
  public_draft_wp(article.title,article.content,article.seo_title,article.meta_description,article.focus_keyphrase)
await main()

Enter project (name or link), project type, and referral link if any:
Solana
